# Notebook pour une baseline LLM closed-book (sans context): Qwen 7B & 14B

# 1. Qwen 7B
## 1) Imports

In [1]:
import os
import json
import ast
import re
from typing import Dict, List

import numpy as np
import pandas as pd
import evaluate
from datasets import load_dataset
from tqdm import tqdm

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)

/home/onyxia/work/RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2) Paramètres (modèle, chemins, tailles)

Cette cellule définit les paramètres principaux.

- **Entrées :** aucune.
- **Ce que fait la cellule :**
  - choisit le modèle (`Qwen 7B`, `Qwen 14B`),
  - donne les chemins des fichiers `train.csv` et `validation.csv`,
  - fixe la graine `SEED` (reproductibilité),
  - fixe les tailles de génération :
    - `MAX_INPUT_TOKENS` : longueur max du prompt (question),
    - `MAX_NEW_TOKENS` : longueur max de la réponse générée,
    - `BATCH_SIZE` : nombre de questions traitées à la fois.
- **Sorties :** aucune (variables globales).

In [9]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
#MODEL_NAME = "JungZoona/T3Q-qwen2.5-14b-v1.0-e3"



DATA_TRAIN = "data/train.csv"
DATA_VALID = "data/validation.csv"

PRED_JSONL = "results/sample_predictions_qwen7B_no_rag.jsonl"
OUT_CSV = "results/manual_eval_qwen7B.xlsx"

SEED = 42


MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS = 32
BATCH_SIZE = 1  
DEVICE = "cuda"

## 3) Nettoyage de la colonne `answers`

Ces fonctions mettent la colonne `answers` au format attendu par la métrique SQuAD.

### `parse_answers_field(x)`
- **Entrée :** `x` (texte) = contenu brut de la colonne `answers`.
- **Ce que fait la fonction :**
  - nettoie le texte (supprime la forme `array(..., dtype=...)`),
  - transforme le texte en dictionnaire Python,
  - convertit en listes Python (`text`, `answer_start`).
- **Sortie :** dictionnaire `{"text": [...], "answer_start": [...]}`.

### `normalize_dataset(example)`
- **Entrée :** une ligne du dataset (dict).
- **Ce que fait la fonction :**
  - applique `parse_answers_field` sur `answers`,
  - convertit `id` en string.
- **Sortie :** la ligne corrigée.

In [3]:
def parse_answers_field(x):
    s = str(x).strip()
    s2 = re.sub(r"array\(\s*(\[[\s\S]*?\])\s*,\s*dtype=[^)]+\)", r"\1", s)
    ans = ast.literal_eval(s2)
    ans["text"] = list(ans["text"])
    ans["answer_start"] = [int(v) for v in ans["answer_start"]]
    return ans

def normalize_dataset(example):
    example["answers"] = parse_answers_field(example["answers"])
    example["id"] = str(example["id"])
    return example

## 4) Construction du prompt et nettoyage de la génération

Ces fonctions servent à dialoguer avec le LLM de façon simple.

### `build_prompt(question, tokenizer)`
- **Entrées :**
  - `question` : la question,
  - `tokenizer` : tokenizer du modèle.
- **Ce que fait la fonction :**
  - construit un petit “rôle système” (répondre court, sans explication),
  - ajoute la question,
  - si le modèle a un *chat template*, l’utilise automatiquement.
- **Sortie :** un texte (prompt) prêt à être tokenisé.

### `clean_generation(text)`
- **Entrée :** texte généré par le modèle.
- **Ce que fait la fonction :**
  - garde la première ligne,
  - enlève des préfixes possibles (`Answer:`, etc.).
- **Sortie :** réponse finale nettoyée.

In [4]:
def build_prompt(question: str, tokenizer) -> str:
    question = str(question).strip()
    system_msg = (
        "You are a helpful assistant. "
        "Answer the question with a short phrase. "
        "Do not add explanations."
    )
    user_msg = f"Question: {question}\nAnswer:"

    if hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    return f"{system_msg}\n\n{user_msg}"

def clean_generation(text: str) -> str:
    t = (text or "").strip()
    
    t = t.split("\n")[0].strip()
    
    for pref in ["Answer:", "A:", "assistant:", "Assistant:"]:
        if t.lower().startswith(pref.lower()):
            t = t[len(pref):].strip()
    return t

## 5) Chargement des données et du modèle (Qwen)

Cette cellule prépare le dataset et charge le LLM.

- **Entrées :**
  - `data/train.csv`
  - `data/validation.csv`
- **Ce que fait la cellule :**
  1) fixe la graine (`set_seed`) pour la reproductibilité,
  2) charge les CSV avec `load_dataset`,
  3) applique `normalize_dataset` pour avoir `answers` au bon format,
  4) charge le tokenizer du modèle,
  5) charge le modèle en quantization (4-bit) pour économiser la VRAM.
- **Sorties :**
  - `ds` : dataset (train + validation),
  - `tokenizer` : tokenizer Qwen,
  - `model` : modèle Qwen prêt pour la génération.

In [7]:
set_seed(SEED)

ds = load_dataset("csv", data_files={"train": DATA_TRAIN, "validation": DATA_VALID})
ds = ds.map(normalize_dataset)



tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token




bnb8 = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb8,
    device_map={"": 0},
    torch_dtype=torch.float16,
)

'''
# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)



model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
)'''

model.eval()


Generating train split: 87599 examples [00:01, 80434.85 examples/s]
Generating validation split: 1000 examples [00:00, 39136.56 examples/s]
Map: 100%|██████████| 1000/1000 [00:00<00:00, 8274.11 examples/s]
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:13<00:00,  3.41s/it]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear8bitLt(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear8bitLt(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear8bitLt(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear8bitLt(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear8bitLt(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear8bitLt(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear8bitLt(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm)

## 6) Génération sur la validation (No-RAG, question seule)

Cette cellule génère une réponse pour chaque question (sans contexte, sans retrieval).

- **Entrées :**
  - `ds["validation"]` (questions + réponses gold),
  - le modèle `model` et le `tokenizer`.
- **Ce que fait la cellule :**
  1) boucle sur les questions validation,
  2) crée un prompt avec `build_prompt(question)`,
  3) tokenise le prompt,
  4) génère la réponse avec `model.generate(...)`,
  5) nettoie la réponse avec `clean_generation`,
  6) stocke le résultat dans `predictions` sous la forme :
     `{"id": ..., "prediction_text": ...}`.
- **Sortie :**
  - `predictions` : liste des prédictions du modèle.

In [8]:
model.config.use_cache = False 
eval_split = ds["validation"]

predictions = []
squad_metric = evaluate.load("squad")

ids = eval_split["id"]
questions = eval_split["question"]

for start in tqdm(range(0, len(eval_split), BATCH_SIZE), desc="Generating (No-RAG, Qwen 7B)"):
    batch_ids = ids[start:start + BATCH_SIZE]
    batch_q = questions[start:start + BATCH_SIZE]

    prompts = [build_prompt(q, tokenizer) for q in batch_q]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )

    with torch.inference_mode():
        out = model.generate(
            **inputs,
            use_cache=False,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    gen_ids = out[:, prompt_len:]
    gen_texts = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

    del inputs, out, gen_ids
    torch.cuda.empty_cache()

    for ex_id, gt in zip(batch_ids, gen_texts):
        predictions.append({"id": str(ex_id), "prediction_text": clean_generation(gt)})

Generating (No-RAG, Qwen 7B):   0%|          | 0/1000 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/home/onyxia/work/RAG/.venv/lib/python3.13/site-packages/transformers/generation/utils.py:2534: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(
Generating (No-RAG, Qwen 7B): 100%|██████████| 1000/1000 [18:08<00:00,  1.09s/it]


## 7) Calcul des métriques (Exact Match et F1)

Cette cellule calcule EM/F1 avec la métrique officielle SQuAD.

- **Entrées :**
  - `predictions` (réponses générées),
  - `eval_split` (réponses gold dans `answers`).
- **Ce que fait la cellule :**
  1) construit `references` au format SQuAD,
  2) calcule `exact_match` et `f1`,
  3) affiche un petit tableau récapitulatif.
- **Sorties :**
  - `results` : dict avec EM/F1,
  - `df` : tableau des résultats.

In [10]:
# ---- Metrics (EM/F1) ----
references = [{"id": str(ex["id"]), "answers": ex["answers"]} for ex in eval_split]
results = squad_metric.compute(predictions=predictions, references=references)

df = pd.DataFrame([{
    "split": "validation",
    "exact_match": results["exact_match"],
    "f1": results["f1"],
    "model": MODEL_NAME,
}])

print("\n=== No-RAG (Qwen 7B) Validation Results ===")
print(df.to_string(index=False))




=== No-RAG (Qwen 7B) Validation Results ===
     split  exact_match        f1                    model
validation         11.4 21.797348 Qwen/Qwen2.5-7B-Instruct


## 8) Sauvegarde des résultats

Cette cellule enregistre les résultats sur disque.

- **Entrées :**
  - `df` (tableau des scores),
  - `results` (dict des métriques),
  - `predictions` (liste des prédictions).
- **Ce que fait la cellule :**
  - crée le dossier `results/` si besoin,
  - sauvegarde :
    - un CSV des scores,
    - un JSON des métriques,
    - un échantillon de prédictions (JSONL).
- **Sorties :**
  - fichiers dans `results/`.

In [12]:
os.makedirs("results", exist_ok=True)
df.to_excel("results/metrics_table_qwen7B_no_rag.xlsx", index=False)


sample_path = "results/sample_predictions_qwen7B_no_rag.jsonl"
with open(sample_path, "w") as f:
    for row in predictions[:200]:
        f.write(json.dumps(row) + "\n")

print("\nSaved:")
print(" - results/metrics_table_qwen7B_no_rag.xlsx")
print(" - results/sample_predictions_qwen7B_no_rag.jsonl")



Saved:
 - results/metrics_table_qwen7B_no_rag.xlsx
 - results/sample_predictions_qwen7B_no_rag.jsonl


## 9) Exemples (Question / Pred / Gold)


In [13]:
import random

val_by_id = {str(ex["id"]): ex for ex in eval_split}
pred_by_id = {p["id"]: p["prediction_text"] for p in predictions}

sample_ids = random.sample(list(pred_by_id.keys()), 10)
for ex_id in sample_ids:
    ex = val_by_id[ex_id]
    gold = ex["answers"]["text"][0] if ex["answers"]["text"] else ""
    print("Q:", ex["question"])
    print("Pred:", pred_by_id[ex_id])
    print("Gold:", gold)
    print("---")

Q: What is a major importance of Southern California in relation to California and the United States?
Pred: Economic and cultural hub
Gold: economic center
---
Q: How did Mongol armies lure enemy groups out of their defensive positions?
Pred: Feigned retreat
Gold: feigned retreat
---
Q: What month and year was Apollo 13 launched?
Pred: April 1970
Gold: April 1970
---
Q: What does the Discovery Museum draw attention to?
Pred: STEM fields
Gold: life on Tyneside
---
Q: Why is the public library known as a people's university?
Pred: Access to free educational resources.
Gold: it is open to all irrespective of age, literacy level and has materials relevant to people of all walks of life
---
Q: When did Jamaa Islamiya renounce violence?
Pred: 1980
Gold: in 2003
---
Q: Where did the Panthers practice for the Super Bowl?
Pred: Orlando, Florida
Gold: San Jose State practice facility
---
Q: What was the Harvard endowment total in 2011?
Pred: 38.1 billion dollars
Gold: $32 billion
---
Q: To calcu

# 10) Création d'un CSV id, question, gold answer, llm answer pour Qwen 7B & 14B

In [14]:
def load_jsonl_preds(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            rows.append(
                {
                    "id": str(obj["id"]),
                    "prediction": clean_generation(obj.get("prediction_text", "")),
                }
            )
    return pd.DataFrame(rows)


def clean_generation(text: str) -> str:
    t = (text or "").strip()
    t = t.split("\n")[0].strip()
    for pref in ["Answer:", "A:", "assistant:", "Assistant:"]:
        if t.lower().startswith(pref.lower()):
            t = t[len(pref):].strip()
    return t




val = pd.read_csv(DATA_VALID)
val["id"] = val["id"].astype(str)
val["answers_parsed"] = val["answers"].apply(parse_answers_field)
val["gold_answer"] = val["answers_parsed"].apply(lambda a: a["text"][0] if a["text"] else "")
val_base = val[["id", "question", "gold_answer"]].copy()


pred = load_jsonl_preds(PRED_JSONL)
df = val_base.merge(pred, on="id", how="inner")
df = df.rename(columns={"prediction": "llm_answer"})
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
df.to_excel(OUT_CSV, index=False)

print("Saved:")
print(" -", OUT_CSV, f"({len(df)} rows)")

Saved:
 - results/manual_eval_qwen7B.xlsx (200 rows)


# 2. Qwen 14B

In [1]:
import os
import json
import ast
import re
from typing import Dict, List

import numpy as np
import pandas as pd
import evaluate
from datasets import load_dataset
from tqdm import tqdm

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)


#MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MODEL_NAME = "JungZoona/T3Q-qwen2.5-14b-v1.0-e3"



DATA_TRAIN = "data/train.csv"
DATA_VALID = "data/validation.csv"

PRED_JSONL = "results/sample_predictions_qwen14B_no_rag.jsonl"
OUT_CSV = "results/manual_eval_qwen14B.xlsx"

SEED = 42


MAX_INPUT_TOKENS = 512
MAX_NEW_TOKENS = 32
BATCH_SIZE = 1  
DEVICE = "cuda"

def parse_answers_field(x):
    s = str(x).strip()
    s2 = re.sub(r"array\(\s*(\[[\s\S]*?\])\s*,\s*dtype=[^)]+\)", r"\1", s)
    ans = ast.literal_eval(s2)
    ans["text"] = list(ans["text"])
    ans["answer_start"] = [int(v) for v in ans["answer_start"]]
    return ans

def normalize_dataset(example):
    example["answers"] = parse_answers_field(example["answers"])
    example["id"] = str(example["id"])
    return example


def build_prompt(question: str, tokenizer) -> str:
    question = str(question).strip()
    system_msg = (
        "You are a helpful assistant. "
        "Answer the question with a short phrase. "
        "Do not add explanations."
    )
    user_msg = f"Question: {question}\nAnswer:"

    if hasattr(tokenizer, "apply_chat_template"):
        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ]
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    return f"{system_msg}\n\n{user_msg}"

def clean_generation(text: str) -> str:
    t = (text or "").strip()
    
    t = t.split("\n")[0].strip()
    
    for pref in ["Answer:", "A:", "assistant:", "Assistant:"]:
        if t.lower().startswith(pref.lower()):
            t = t[len(pref):].strip()
    return t


set_seed(SEED)

ds = load_dataset("csv", data_files={"train": DATA_TRAIN, "validation": DATA_VALID})
ds = ds.map(normalize_dataset)



tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token



'''
bnb8 = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb8,
    device_map={"": 0},
    torch_dtype=torch.float16,
)
'''


# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)



model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
)

model.eval()



model.config.use_cache = False 
eval_split = ds["validation"]

predictions = []
squad_metric = evaluate.load("squad")

ids = eval_split["id"]
questions = eval_split["question"]

for start in tqdm(range(0, len(eval_split), BATCH_SIZE), desc="Generating (No-RAG, Qwen 14B)"):
    batch_ids = ids[start:start + BATCH_SIZE]
    batch_q = questions[start:start + BATCH_SIZE]

    prompts = [build_prompt(q, tokenizer) for q in batch_q]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )

    with torch.inference_mode():
        out = model.generate(
            **inputs,
            use_cache=False,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]
    gen_ids = out[:, prompt_len:]
    gen_texts = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

    del inputs, out, gen_ids
    torch.cuda.empty_cache()

    for ex_id, gt in zip(batch_ids, gen_texts):
        predictions.append({"id": str(ex_id), "prediction_text": clean_generation(gt)})


# ---- Metrics (EM/F1) ----
references = [{"id": str(ex["id"]), "answers": ex["answers"]} for ex in eval_split]
results = squad_metric.compute(predictions=predictions, references=references)

df = pd.DataFrame([{
    "split": "validation",
    "exact_match": results["exact_match"],
    "f1": results["f1"],
    "model": MODEL_NAME,
}])

print("\n=== No-RAG (Qwen 14B) Validation Results ===")
print(df.to_string(index=False))




os.makedirs("results", exist_ok=True)
df.to_excel("results/metrics_table_qwen14B_no_rag.xlsx", index=False)


sample_path = "results/sample_predictions_qwen14B_no_rag.jsonl"
with open(sample_path, "w") as f:
    for row in predictions[:200]:
        f.write(json.dumps(row) + "\n")

print("\nSaved:")
print(" - results/metrics_table_qwen14B_no_rag.csv")
print(" - results/sample_predictions_qwen14B_no_rag.jsonl")



import random

val_by_id = {str(ex["id"]): ex for ex in eval_split}
pred_by_id = {p["id"]: p["prediction_text"] for p in predictions}

sample_ids = random.sample(list(pred_by_id.keys()), 10)
for ex_id in sample_ids:
    ex = val_by_id[ex_id]
    gold = ex["answers"]["text"][0] if ex["answers"]["text"] else ""
    print("Q:", ex["question"])
    print("Pred:", pred_by_id[ex_id])
    print("Gold:", gold)
    print("---")

def load_jsonl_preds(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            rows.append(
                {
                    "id": str(obj["id"]),
                    "prediction": clean_generation(obj.get("prediction_text", "")),
                }
            )
    return pd.DataFrame(rows)


def clean_generation(text: str) -> str:
    t = (text or "").strip()
    t = t.split("\n")[0].strip()
    for pref in ["Answer:", "A:", "assistant:", "Assistant:"]:
        if t.lower().startswith(pref.lower()):
            t = t[len(pref):].strip()
    return t




val = pd.read_csv(DATA_VALID)
val["id"] = val["id"].astype(str)
val["answers_parsed"] = val["answers"].apply(parse_answers_field)
val["gold_answer"] = val["answers_parsed"].apply(lambda a: a["text"][0] if a["text"] else "")
val_base = val[["id", "question", "gold_answer"]].copy()


pred = load_jsonl_preds(PRED_JSONL)
df = val_base.merge(pred, on="id", how="inner")
df = df.rename(columns={"prediction": "llm_answer"})
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
df.to_excel(OUT_CSV, index=False)

print("Saved:")
print(" -", OUT_CSV, f"({len(df)} rows)")
    

/home/onyxia/work/RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]/home/onyxia/work/RAG/.venv/lib/python3.13/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Generating (No-RAG, Qwen 14B):   0%|          | 0/1000 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/home/onyxia/work/RAG/.venv/lib/python3.13/site-packages/transformers/generation/utils.py:2534: UserWarning: You are calling .generate() with the `input_ids` being on a device type d


=== No-RAG (Qwen 14B) Validation Results ===
     split  exact_match        f1                             model
validation         15.2 27.298001 JungZoona/T3Q-qwen2.5-14b-v1.0-e3

Saved:
 - results/metrics_table_qwen14B_no_rag.csv
 - results/sample_predictions_qwen14B_no_rag.jsonl
Q: What is a major importance of Southern California in relation to California and the United States?
Pred: Economic hub
Gold: economic center
---
Q: How did Mongol armies lure enemy groups out of their defensive positions?
Pred: By using feigned retreats and decoys.
Gold: feigned retreat
---
Q: What month and year was Apollo 13 launched?
Pred: April 1970
Gold: April 1970
---
Q: What does the Discovery Museum draw attention to?
Pred: The Discovery Museum draws attention to science, technology, and innovation through interactive exhibits and educational programs.
Gold: life on Tyneside
---
Q: Why is the public library known as a people's university?
Pred: Because it provides free access to knowledge and lea